In [1]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import seaborn as sns
from matplotlib.image import imread
from PIL import Image
import tensorflow as tf
np.random.seed(1337)
import gc

import glob

from tensorflow.keras import layers
from keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.models import Sequential,Model
from tensorflow.keras.layers import Input, Activation,Conv2DTranspose, Dropout, AveragePooling2D,Flatten, Dense, Conv2D,MaxPool2D, MaxPooling2D, BatchNormalization,Conv2DTranspose,concatenate,UpSampling2D
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')


img_size = 128
print(os.listdir())
dataset = os.listdir("music_seperation_dataset/train")
labels = dataset
print(labels)

['.git', '.vscode', 'drum_part.wav', 'drum_prediction.wav', 'full.wav', 'model_classes_3.ipynb', 'model_classes_full.ipynb', 'model_separation.ipynb', 'music_dataset_spectro_3_instrument', 'music_dataset_spectro_full', 'music_seperation_dataset', 'my_model_3.keras', 'my_model_full.keras', 'my_separator_model_prototype.keras', 'README.md', 'separated_part.png', 'separated_part_1_batch.png', 'spectrogramMaker.py']
['Acoustic_Guitar', 'Bass_Guitar', 'Drum_set', 'Electric_Guitar', 'full_mix', 'Keyboard']


In [2]:
def get_dataset_array(data_dir):
    data = []
    path = os.path.join(data_dir)
    for img in os.listdir(data_dir):
            try:
                img_arr = cv2.imread(os.path.join(path,img))
                resized_arr = cv2.resize(img_arr, (img_size, img_size))
                data.append([resized_arr])
                gc.collect()
            except Exception as e:
                print(e)
    return np.array(data,dtype="float32") 

In [3]:

x_train = get_dataset_array("music_seperation_dataset/train/full_mix")
y_train = get_dataset_array("music_seperation_dataset/train/Drum_set")

x_test = get_dataset_array("music_seperation_dataset/test/full_mix")
y_test = get_dataset_array("music_seperation_dataset/test/Drum_set")

x_valid = get_dataset_array("music_seperation_dataset/valid/full_mix")
y_valid = get_dataset_array("music_seperation_dataset/valid/Drum_set")



In [4]:
gc.collect()
x_train = np.array(x_train)/255
gc.collect()
x_test = np.array(x_test)/255
gc.collect()
x_valid = np.array(x_valid)/255
gc.collect()
y_train = np.array(y_train)/255
gc.collect()
y_test = np.array(y_test)/255
gc.collect()
y_valid = np.array(y_valid)/255
gc.collect()

0

In [5]:
x_train = x_train.reshape(-1, img_size, img_size, 1)
y_train = y_train.reshape(-1, img_size, img_size, 1)

x_valid = x_valid.reshape(-1, img_size, img_size, 1)
y_valid = y_valid.reshape(-1, img_size, img_size, 1)

x_test = x_test.reshape(-1, img_size, img_size, 1)
y_test = y_test.reshape(-1, img_size, img_size, 1)


In [19]:
num_classes = 1
def Unet():
    inputs =  layers.Input(shape=(None,None,1))

    conv1 = Conv2D(16, (3,3), activation = 'relu', padding='same')(inputs)
    conv1 = BatchNormalization()(conv1)
    conv1 = Conv2D(16, (3,3), activation = 'relu', padding='same')(conv1)
    conv1 = BatchNormalization()(conv1)
    pool1 = MaxPool2D((2,2))(conv1)

    conv2 = Conv2D(32, (3,3), activation = 'relu', padding='same')(pool1)
    conv2 = BatchNormalization()(conv2)
    conv2 = Conv2D(32, (3,3), activation = 'relu', padding='same')(conv2)
    conv2 = BatchNormalization()(conv2)
    pool2 = MaxPool2D((2,2))(conv2)

    conv3 = Conv2D(64, (3,3), activation = 'relu', padding='same')(pool2)
    conv3 = BatchNormalization()(conv3)
    conv3 = Conv2D(64, (3,3), activation = 'relu', padding='same')(conv3)
    conv3 = BatchNormalization()(conv3)
    pool3 = MaxPool2D((2,2))(conv3)

    conv4 = Conv2D(128, (3,3), activation = 'relu', padding='same')(pool3)
    conv4 = BatchNormalization()(conv4)
    conv4 = Conv2D(128, (3,3), activation = 'relu', padding='same')(conv4)
    conv4 = BatchNormalization()(conv4)
    pool4 = MaxPool2D((2,2))(conv4)

    conv5 = Conv2D(256, (3,3), activation = 'relu', padding='same')(pool4)
    conv5 = BatchNormalization()(conv5)
    conv5 = Conv2D(256, (3,3), activation = 'relu', padding='same')(conv5)
    conv5 = BatchNormalization()(conv5)
    
    goUp1 = UpSampling2D((2,2))(conv5)
    goUp1 = Conv2D(128,(3,3),padding='same',activation = 'relu',)(goUp1)
    goUp1 = BatchNormalization()(goUp1)
    goUp1 = concatenate([goUp1,conv4])
    conv6 = Conv2D(128, (3,3), activation = 'relu', padding='same')(goUp1)
    conv6 = BatchNormalization()(conv6)
    conv6 = Conv2D(128, (3,3), activation = 'relu', padding='same')(conv6)
    conv6 = BatchNormalization()(conv6)

    goUp2 = UpSampling2D((2,2))(conv6)
    goUp2 = Conv2D(64,(3,3),padding='same',activation = 'relu',)(goUp2)
    goUp2 = BatchNormalization()(goUp2)
    goUp2 = concatenate([goUp2,conv3])
    conv7 = Conv2D(64, (3,3), activation = 'relu', padding='same')(goUp2)
    conv7 = BatchNormalization()(conv7)
    conv7 = Conv2D(64, (3,3), activation = 'relu', padding='same')(conv7)
    conv7 = BatchNormalization()(conv7)

    goUp3 = UpSampling2D((2,2))(conv7)
    goUp3 = Conv2D(32,(3,3),padding='same',activation = 'relu',)(goUp3)
    goUp3 = concatenate([goUp3,conv2])
    goUp3 = BatchNormalization()(goUp3)
    conv8 = Conv2D(32, (3,3), activation = 'relu', padding='same')(goUp3)
    conv8 = BatchNormalization()(conv8)
    conv8 = Conv2D(32, (3,3), activation = 'relu', padding='same',)(conv8)
    conv8 = BatchNormalization()(conv8)

    goUp4 = UpSampling2D((2,2))(conv8)
    goUp4 = Conv2D(16,(3,3),padding='same',activation = 'relu',)(goUp4)
    goUp4 = BatchNormalization()(goUp4)
    goUp4 = concatenate([goUp4,conv1])
    conv9 = Conv2D(16, (3,3), activation = 'relu', padding='same')(goUp4)
    conv9 = BatchNormalization()(conv9)
    conv9 = Conv2D(16, (3,3), activation = 'relu', padding='same')(conv9)
    conv9 = BatchNormalization()(conv9)



    outputs = Conv2D(num_classes,(1,1),activation="sigmoid")(conv9)

    model = Model(inputs=[inputs],outputs=[outputs])
    return model

model = Unet()
model.compile(
              optimizer = 'adam', loss = 'mse',
              metrics = ['mae']
              )
     

In [20]:
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ None, 1)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_46 (Conv2D)  │ (None, None,      │        160 │ input_layer_2[0]… │
│                     │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, None,      │         64 │ conv2d_46[0][0]   │
│ (BatchNormalizatio… │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_47 (Conv2D)  │ (None, None,      │      2,320 │ batch_normalizat… │
│                     │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │         64 │ conv2d_47[0][0]   │
│ (BatchNormalizatio… │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_8     │ (None, None,      │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_48 (Conv2D)  │ (None, None,      │      4,640 │ max_pooling2d_8[… │
│                     │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │        128 │ conv2d_48[0][0]   │
│ (BatchNormalizatio… │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_49 (Conv2D)  │ (None, None,      │      9,248 │ batch_normalizat… │
│                     │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │        128 │ conv2d_49[0][0]   │
│ (BatchNormalizatio… │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_9     │ (None, None,      │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_50 (Conv2D)  │ (None, None,      │     18,496 │ max_pooling2d_9[… │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │        256 │ conv2d_50[0][0]   │
│ (BatchNormalizatio… │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_51 (Conv2D)  │ (None, None,      │     36,928 │ batch_normalizat… │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │        256 │ conv2d_51[0][0]   │
│ (BatchNormalizatio… │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_10    │ (None, None,      │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_52 (Conv2D)  │ (None, None,      │     73,856 │ max_pooling2d_10

 Total params: 2,165,393 (8.26 MB)

 Trainable params: 2,161,905 (8.25 MB)

 Non-trainable params: 3,488 (13.62 KB)

In [21]:
batch_size = 1
n_epochs = 100
model.fit(x_train, y_train, batch_size = batch_size,
                    epochs = n_epochs, validation_data = (x_valid, y_valid))

Epoch 1/100
144/144 ━━━━━━━━━━━━━━━━━━━━ 17s 54ms/step - loss: 0.0155 - mae: 0.0872 - val_loss: 0.0536 - val_mae: 0.1908
Epoch 2/100
144/144 ━━━━━━━━━━━━━━━━━━━━ 7s 50ms/step - loss: 0.0050 - mae: 0.0498 - val_loss: 0.0440 - val_mae: 0.1706
Epoch 3/100
144/144 ━━━━━━━━━━━━━━━━━━━━ 7s 48ms/step - loss: 0.0042 - mae: 0.0448 - val_loss: 0.0243 - val_mae: 0.1147
Epoch 4/100
144/144 ━━━━━━━━━━━━━━━━━━━━ 7s 49ms/step - loss: 0.0038 - mae: 0.0421 - val_loss: 0.0115 - val_mae: 0.0790
Epoch 5/100
144/144 ━━━━━━━━━━━━━━━━━━━━ 7s 49ms/step - loss: 0.0034 - mae: 0.0393 - val_loss: 0.0091 - val_mae: 0.0700
Epoch 6/100
144/144 ━━━━━━━━━━━━━━━━━━━━ 7s 48ms/step - loss: 0.0033 - mae: 0.0387 - val_loss: 0.0096 - val_mae: 0.0673
Epoch 7/100
144/144 ━━━━━━━━━━━━━━━━━━━━ 7s 49ms/step - loss: 0.0031 - mae: 0.0374 - val_loss: 0.0096 - val_mae: 0.0680
Epoch 8/100
144/144 ━━━━━━━━━━━━━━━━━━━━ 7s 48ms/step - loss: 0.0031 - mae: 0.0374 - val_loss: 0.0094 - val_mae: 0.0688
Epoch 9/100
144/144 ━━━━━━━━━━━━━━━━━━━

In [9]:
gc.collect()

model.save('my_separator_model_prototype.keras')

In [89]:
img = cv2.imread("music_seperation_dataset/test/full_mix/53_full_mix.png",0)

img=cv2.resize(img,(128,128))
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255

predimg= np.squeeze(model.predict(img))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part.png",prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
(128, 128)


True

In [90]:
import librosa

image = cv2.imread("separated_part.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=64,sr=22050)


audio = librosa.util.normalize(audio)

import soundfile
soundfile.write('drum_prediction.wav',audio, 22050)




In [ ]:
image = cv2.imread("music_seperation_dataset/test/full_mix/53_full_mix.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)

import soundfile
soundfile.write('full.wav',audio, 22050)

In [13]:
image = cv2.imread("music_seperation_dataset/test/Drum_set/53_Drum_set.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=1000,sr=22050)


audio = librosa.util.normalize(audio)

import soundfile
soundfile.write('drum_part.wav',audio, 22050)

In [85]:
img = cv2.imread("music_dataset_spectro_full/valid/songsfull/0_songsfull.png",0)

img=cv2.resize(img,(256,6016))
img= np.reshape(img,(-1, 256, 6016, 1))
img=img/255

predimg= np.squeeze(model.predict(img))

prediction= predimg*255
print(prediction.shape)



cv2.imwrite("separated_part.png",prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 692ms/step
(256, 6016)


True

In [ ]:
import librosa

image = cv2.imread("music_dataset_spectro_full/valid/songsfull/0_songsfull.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=32,sr=22050,power=1)

audio = librosa.util.normalize(audio)

import soundfile
soundfile.write('song.wav',audio, 22050)

In [ ]:
freq, sr = librosa.load("music_dataset/songsfull/14.wav")

short_time= librosa.stft(freq)

mag,phase = librosa.magphase(short_time)

print(phase.shape)
print(mag.shape)
mel = librosa.feature.melspectrogram(sr=sr,S=mag)

inverse_mel = librosa.feature.inverse.mel_to_stft(mel,sr=sr)


short_inverse = mag * np.exp(phase)

recon = librosa.istft(short_inverse)
recon = librosa.util.normalize(recon)

soundfile.write('song.wav',recon, 22050)


(1025, 6014)
(1025, 6014)
(128, 6014)
